## Audit new drift-aware backtest engine

### Setup

In [1]:
import numpy as np
import pandas as pd

from alpha_research.backtest import (
    BacktestConfig,
    run_long_short_backtest,
    run_target_weight_backtest,
    summarise_backtest,
)
from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.data_loader import load_parquet
from alpha_research.portfolio import build_factor_target_weights


panel = load_parquet(
    PROCESSED_DATA_DIR / "factor_panel.parquet"
)

factor_columns = {
    "12-1 Momentum": "mom_12_1m_z",
    "Realised Volatility": "realised_vol_63_z",
}

config = BacktestConfig(
    rebalance_frequency=5,
    quantiles=5,
    long_quantile=5,
    short_quantile=1,
    long_gross=1.0,
    short_gross=1.0,
    transaction_cost_bps=10.0,
    min_observations=30,
    rebalance_offset=0,
)

### Run both engines

In [2]:
audit_results = {}

return_panel = panel[
    ["date", "ticker", "forward_ret_1d"]
].copy()

for factor_name, factor_column in factor_columns.items():
    target_weights = build_factor_target_weights(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    legacy_daily, legacy_holdings = run_long_short_backtest(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    drift_daily, drift_holdings = run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=target_weights,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )

    audit_results[factor_name] = {
        "targets": target_weights,
        "legacy_daily": legacy_daily,
        "legacy_holdings": legacy_holdings,
        "drift_daily": drift_daily,
        "drift_holdings": drift_holdings,
    }

### Verify the results

Both engines should match the target weights exactly on rebalance dates.

In [3]:
def maximum_weight_difference(
    left: pd.DataFrame,
    right: pd.DataFrame,
) -> float:
    comparison = left[["date", "ticker", "weight"]].merge(
        right[["date", "ticker", "weight"]],
        on=["date", "ticker"],
        how="outer",
        suffixes=("_left", "_right"),
    )

    comparison[["weight_left", "weight_right"]] = comparison[
        ["weight_left", "weight_right"]
    ].fillna(0.0)

    return float((comparison["weight_left"] - comparison["weight_right"]).abs().max())


audit_check_rows = []

for factor_name, result in audit_results.items():
    targets = result["targets"]
    target_dates = targets["date"].unique()

    legacy_rebalance_holdings = result["legacy_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    drift_rebalance_holdings = result["drift_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    daily_comparison = result["legacy_daily"][
        ["date", "is_rebalance", "gross_return"]
    ].merge(
        result["drift_daily"][["date", "is_rebalance", "gross_return"]],
        on="date",
        how="inner",
        suffixes=("_legacy", "_drift"),
        validate="one_to_one",
    )

    gross_difference = (
        daily_comparison["gross_return_drift"] - daily_comparison["gross_return_legacy"]
    ).abs()

    rebalance_mask = daily_comparison["is_rebalance_legacy"]

    audit_check_rows.append(
        {
            "factor": factor_name,
            "same_daily_dates": (
                len(daily_comparison)
                == len(result["legacy_daily"])
                == len(result["drift_daily"])
            ),
            "rebalance_dates": len(target_dates),
            "max_legacy_target_mismatch": (
                maximum_weight_difference(
                    legacy_rebalance_holdings,
                    targets,
                )
            ),
            "max_drift_target_mismatch": (
                maximum_weight_difference(
                    drift_rebalance_holdings,
                    targets,
                )
            ),
            "max_rebalance_gross_return_difference": (
                gross_difference.loc[rebalance_mask].max()
            ),
            "mean_non_rebalance_gross_return_difference": (
                gross_difference.loc[~rebalance_mask].mean()
            ),
        }
    )

audit_checks = pd.DataFrame(audit_check_rows).set_index("factor")

audit_checks

,same_daily_dates,rebalance_dates,max_legacy_target_mismatch,max_drift_target_mismatch,max_rebalance_gross_return_difference,mean_non_rebalance_gross_return_difference
factor,,,,,,
12-1 Momentum,True,578,0.0,0.0,0.0,0.000349
Realised Volatility,True,578,0.0,0.0,0.0,0.000262


#### Compare economic results

In [4]:
summary_rows = []

for factor_name, result in audit_results.items():
    for engine_name, daily in {
        "Legacy": result["legacy_daily"],
        "Drift-aware": result["drift_daily"],
    }.items():
        gross = summarise_backtest(
            daily,
            return_column="gross_return",
        ).iloc[0]

        net = summarise_backtest(
            daily,
            return_column="net_return",
        ).iloc[0]

        summary_rows.append(
            {
                "factor": factor_name,
                "engine": engine_name,
                "gross_total_return": gross["total_return"],
                "net_total_return": net["total_return"],
                "gross_annualised_return": (gross["annualised_return"]),
                "net_annualised_return": (net["annualised_return"]),
                "net_annualised_volatility": (net["annualised_volatility"]),
                "gross_sharpe": gross["sharpe_ratio"],
                "net_sharpe": net["sharpe_ratio"],
                "average_rebalance_turnover": (net["average_rebalance_turnover"]),
                "total_transaction_cost": (net["total_transaction_cost"]),
            }
        )

audit_summary = (
    pd.DataFrame(summary_rows).sort_values(["factor", "engine"]).reset_index(drop=True)
)

audit_summary.round(4)

,factor,engine,gross_total_return,net_total_return,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,average_rebalance_turnover,total_transaction_cost
0,12-1 Momentum,Drift-aware,0.4804,0.1038,0.0348,0.0086,0.2119,0.2683,0.1475,0.5080,0.2936
1,12-1 Momentum,Legacy,0.5957,0.2379,0.0416,0.0188,0.2113,0.2995,0.1947,0.4393,0.2539
2,Realised Volatility,Drift-aware,5.5496,4.1786,0.1781,0.1542,0.2395,0.8042,0.7185,0.4063,0.2348
3,Realised Volatility,Legacy,5.5559,4.3711,0.1782,0.1579,0.2399,0.8036,0.7310,0.3448,0.1993


#### Isolate the change

In [5]:
metric_columns = [
    "gross_annualised_return",
    "net_annualised_return",
    "net_annualised_volatility",
    "gross_sharpe",
    "net_sharpe",
    "average_rebalance_turnover",
    "total_transaction_cost",
]

legacy_summary = audit_summary.loc[audit_summary["engine"] == "Legacy"].set_index(
    "factor"
)

drift_summary = audit_summary.loc[audit_summary["engine"] == "Drift-aware"].set_index(
    "factor"
)

audit_deltas = drift_summary[metric_columns] - legacy_summary[metric_columns]

audit_deltas.columns = [f"change_in_{column}" for column in audit_deltas.columns]

audit_deltas.round(4)

,change_in_gross_annualised_return,change_in_net_annualised_return,change_in_net_annualised_volatility,change_in_gross_sharpe,change_in_net_sharpe,change_in_average_rebalance_turnover,change_in_total_transaction_cost
factor,,,,,,,
12-1 Momentum,-0.0068,-0.0101,0.0006,-0.0312,-0.0472,0.0687,0.0397
Realised Volatility,-0.0001,-0.0037,-0.0004,0.0006,-0.0125,0.0615,0.0355


### Engine audit conclusion

The drift-aware engine reproduces the legacy target portfolios exactly on rebalance dates. Rebalance-day gross returns also match exactly, confirming that the factor signals, stock selection, and target-weight construction are unchanged.

Between rebalances, the drift-aware engine carries forward holdings whose weights evolve with asset returns. This produces non-zero return differences relative to the legacy constant-weight convention and increases measured rebalance turnover.

The legacy implementation understated average rebalance turnover by approximately 0.069 for momentum and 0.062 for realised volatility. After realistic drift accounting, momentum's net Sharpe declines from approximately 0.195 to 0.148, while realised volatility's net Sharpe declines from 0.731 to 0.719.

All subsequent portfolio experiments will therefore use the drift-aware target-weight engine. Legacy results will be retained only as historical benchmarks.

## First multi-factor baseline

A 50/50 combination of momentum and realised volatility, independent sleeves

In [6]:
from alpha_research.portfolio import (
    build_factor_target_weights,
    combine_sleeve_target_weights,
)

In [7]:
momentum_targets = audit_results["12-1 Momentum"]["targets"]

volatility_targets = audit_results["Realised Volatility"]["targets"]

combined_targets = combine_sleeve_target_weights(
    sleeve_targets={
        "Momentum": momentum_targets,
        "Realised Volatility": volatility_targets,
    },
    sleeve_allocations={
        "Momentum": 0.5,
        "Realised Volatility": 0.5,
    },
)

combined_daily, combined_holdings = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=combined_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=config.transaction_cost_bps,
)

#### Examine natural netting

In [8]:
combined_target_exposure = (
    combined_targets.assign(
        long_weight=lambda df: df["weight"].clip(lower=0.0),
        short_weight=lambda df: -df["weight"].clip(upper=0.0),
        active_position=lambda df: df["weight"].ne(0.0).astype(int),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
        active_positions=("active_position", "sum"),
    )
)

combined_target_exposure["gross_exposure"] = (
    combined_target_exposure["long_exposure"]
    + combined_target_exposure["short_exposure"]
)

combined_target_exposure["net_exposure"] = (
    combined_target_exposure["long_exposure"]
    - combined_target_exposure["short_exposure"]
)

combined_target_exposure[
    [
        "long_exposure",
        "short_exposure",
        "gross_exposure",
        "net_exposure",
        "active_positions",
    ]
].agg(["mean", "std", "min", "max"]).round(4)

,long_exposure,short_exposure,gross_exposure,net_exposure,active_positions
mean,0.7352,0.7352,1.4704,0.0,49.9204
std,0.1905,0.1905,0.3810,0.0,10.8236
min,0.0000,0.0000,0.0000,0.0,0.0000
max,1.0000,1.0000,2.0000,0.0,66.0000


#### Compare the three portfolios

In [9]:
baseline_daily = {
    "Momentum": audit_results["12-1 Momentum"]["drift_daily"],
    "Realised Volatility": audit_results["Realised Volatility"]["drift_daily"],
    "50/50 Independent Sleeves": combined_daily,
}

baseline_rows = []

for portfolio_name, daily in baseline_daily.items():
    gross = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    baseline_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross["annualised_return"],
            "net_annualised_return": net["annualised_return"],
            "net_annualised_volatility": net["annualised_volatility"],
            "gross_sharpe": gross["sharpe_ratio"],
            "net_sharpe": net["sharpe_ratio"],
            "max_drawdown": net["max_drawdown"],
            "average_rebalance_turnover": net["average_rebalance_turnover"],
            "total_transaction_cost": net["total_transaction_cost"],
            "average_gross_exposure": daily["gross_exposure"].mean(),
        }
    )

baseline_summary = pd.DataFrame(baseline_rows).set_index("portfolio")

baseline_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
50/50 Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712


In [10]:
gross_return_comparison = pd.concat(
    {
        name: daily.set_index("date")["gross_return"]
        for name, daily in baseline_daily.items()
    },
    axis=1,
)

gross_return_comparison.corr().round(4)

,Momentum,Realised Volatility,50/50 Independent Sleeves
Momentum,1.0000,0.0225,0.6696
Realised Volatility,0.0225,1.0000,0.7572
50/50 Independent Sleeves,0.6696,0.7572,1.0000


### Experiment 1: 50/50 independent factor sleeves

The independent-sleeve portfolio allocates 50% of notional capital to the momentum portfolio and 50% to the realised-volatility portfolio, then nets their stock-level target weights.

The standalone factor returns have very low correlation (approximately 0.02), providing meaningful diversification. Factor disagreement reduces average gross exposure to approximately 1.47, while dollar neutrality is preserved.

The combined portfolio produces a net annualised return of 9.82%, volatility of 16.16%, and a net Sharpe ratio of 0.661. Its maximum drawdown of -25.66% is substantially smaller than the drawdowns of either standalone factor.

The portfolio does not exceed the realised-volatility factor's standalone Sharpe ratio, but it delivers a materially smoother risk profile through diversification and natural position netting.

## Composite score portfolio

Composite score = average of the two factor scores

In [11]:
from alpha_research.portfolio import combine_factor_scores

#### Construct composite score

In [12]:
composite_panel = panel.copy()

composite_panel["mom_vol_composite_z"] = combine_factor_scores(
    panel=composite_panel,
    factor_weights={
        "mom_12_1m_z": 0.5,
        "realised_vol_63_z": 0.5,
    },
)

composite_targets = build_factor_target_weights(
    panel=composite_panel,
    factor_column="mom_vol_composite_z",
    return_column="forward_ret_1d",
    config=config,
)

composite_daily, composite_holdings = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=composite_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=config.transaction_cost_bps,
)

#### Examine target exposure

In [13]:
composite_target_exposure = (
    composite_targets.assign(
        long_weight=lambda df: df["weight"].clip(lower=0.0),
        short_weight=lambda df: -df["weight"].clip(upper=0.0),
        active_position=lambda df: df["weight"].ne(0.0).astype(int),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
        active_positions=("active_position", "sum"),
    )
)

composite_target_exposure["gross_exposure"] = (
    composite_target_exposure["long_exposure"]
    + composite_target_exposure["short_exposure"]
)

composite_target_exposure["net_exposure"] = (
    composite_target_exposure["long_exposure"]
    - composite_target_exposure["short_exposure"]
)

composite_target_exposure.agg(
    ["mean", "std", "min", "max"]
).round(4)

,long_exposure,short_exposure,active_positions,gross_exposure,net_exposure
mean,0.9118,0.9118,36.4706,1.8235,0.0
std,0.2839,0.2839,11.3553,0.5678,0.0
min,0.0000,0.0000,0.0000,0.0000,0.0
max,1.0000,1.0000,40.0000,2.0000,0.0


#### Compare portfolios

In [14]:
comparison_daily = {
    "Momentum": audit_results["12-1 Momentum"]["drift_daily"],
    "Realised Volatility": audit_results["Realised Volatility"]["drift_daily"],
    "50/50 Independent Sleeves": combined_daily,
    "50/50 Composite Score": composite_daily,
}

comparison_rows = []

for portfolio_name, daily in comparison_daily.items():
    gross = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    comparison_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross["annualised_return"],
            "net_annualised_return": net["annualised_return"],
            "net_annualised_volatility": net["annualised_volatility"],
            "gross_sharpe": gross["sharpe_ratio"],
            "net_sharpe": net["sharpe_ratio"],
            "max_drawdown": net["max_drawdown"],
            "average_rebalance_turnover": net["average_rebalance_turnover"],
            "total_transaction_cost": net["total_transaction_cost"],
            "average_gross_exposure": daily["gross_exposure"].mean(),
        }
    )

comparison_summary = pd.DataFrame(comparison_rows).set_index("portfolio")

comparison_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
50/50 Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
50/50 Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [15]:
gross_return_comparison = pd.concat(
    {
        name: daily.set_index("date")["gross_return"]
        for name, daily in comparison_daily.items()
    },
    axis=1,
)

gross_return_comparison.corr().round(4)

,Momentum,Realised Volatility,50/50 Independent Sleeves,50/50 Composite Score
Momentum,1.0000,0.0225,0.6696,0.5510
Realised Volatility,0.0225,1.0000,0.7572,0.7886
50/50 Independent Sleeves,0.6696,0.7572,1.0000,0.9457
50/50 Composite Score,0.5510,0.7886,0.9457,1.0000


### Experiment 2: 50/50 composite factor score

The composite-score portfolio averages the standardised momentum and realised-volatility scores before ranking stocks and constructing a single long-short portfolio.

Unlike the independent-sleeve method, factor disagreement changes stock rankings rather than directly cancelling positions. The portfolio therefore maintains a higher average gross exposure of approximately 1.82.

The composite produces a net annualised return of 12.45%, volatility of 20.37%, and a net Sharpe ratio of 0.679. It modestly exceeds the independent sleeve portfolio's Sharpe ratio, but has higher turnover, transaction costs, and maximum drawdown.

The two combination methods have a return correlation of approximately 0.95, showing that they primarily express the same underlying factor information through different portfolio-construction rules.

## Controlled exposure experiment

If both combination methods carry exactly the same target gross exposure on every rebalance date, does the composite score still perform better?

In [16]:
from alpha_research.portfolio import (
    build_factor_target_weights,
    combine_factor_scores,
    combine_sleeve_target_weights,
    rescale_target_weights_to_gross,
)

In [17]:
composite_target_gross_schedule = (
    composite_targets
    .groupby("date")["weight"]
    .agg(lambda weights: weights.abs().sum())
    .rename("composite_target_gross")
)

controlled_sleeve_targets = rescale_target_weights_to_gross(
    target_weights=combined_targets,
    target_gross=composite_target_gross_schedule,
)

controlled_sleeve_daily, controlled_sleeve_holdings = (
    run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=controlled_sleeve_targets,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )
)

#### Verify exposure control

In [18]:
def calculate_target_gross(
    targets: pd.DataFrame,
) -> pd.Series:
    return (
        targets.groupby("date")["weight"]
        .agg(lambda weights: weights.abs().sum())
    )


target_gross_audit = pd.concat(
    {
        "Original Independent Sleeves": calculate_target_gross(
            combined_targets
        ),
        "Equal-Exposure Independent Sleeves": calculate_target_gross(
            controlled_sleeve_targets
        ),
        "Composite Score": calculate_target_gross(
            composite_targets
        ),
    },
    axis=1,
).fillna(0.0)

target_gross_audit["controlled_minus_composite"] = (
    target_gross_audit["Equal-Exposure Independent Sleeves"]
    - target_gross_audit["Composite Score"]
)

target_gross_audit.agg(
    ["mean", "std", "min", "max"]
).round(6)

,Original Independent Sleeves,Equal-Exposure Independent Sleeves,Composite Score,controlled_minus_composite
mean,1.470415,1.823529,1.823529,0.0
std,0.381037,0.567765,0.567765,0.0
min,0.000000,0.000000,0.000000,-0.0
max,2.000000,2.000000,2.000000,0.0


#### Compare performance

In [19]:
controlled_comparison_daily = {
    "Original Independent Sleeves": combined_daily,
    "Equal-Exposure Independent Sleeves": controlled_sleeve_daily,
    "Composite Score": composite_daily,
}

controlled_summary_rows = []

for portfolio_name, daily in controlled_comparison_daily.items():
    gross_summary = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    controlled_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross_summary[
                "annualised_return"
            ],
            "net_annualised_return": net_summary[
                "annualised_return"
            ],
            "net_annualised_volatility": net_summary[
                "annualised_volatility"
            ],
            "gross_sharpe": gross_summary["sharpe_ratio"],
            "net_sharpe": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "average_rebalance_turnover": net_summary[
                "average_rebalance_turnover"
            ],
            "total_transaction_cost": net_summary[
                "total_transaction_cost"
            ],
            "average_daily_gross_exposure": daily[
                "gross_exposure"
            ].mean(),
        }
    )

controlled_summary = (
    pd.DataFrame(controlled_summary_rows)
    .set_index("portfolio")
)

controlled_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,
Original Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
Equal-Exposure Independent Sleeves,0.1553,0.1224,0.1999,0.8228,0.6779,-0.3173,0.5742,0.3319,1.8240
Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [20]:
matched_exposure_delta = (
    controlled_summary.loc["Composite Score"]
    - controlled_summary.loc[
        "Equal-Exposure Independent Sleeves"
    ]
).rename("composite_minus_equal_exposure_sleeves")

matched_exposure_delta.round(4)

gross_annualised_return        -0.0034
net_annualised_return           0.0022
net_annualised_volatility       0.0038
gross_sharpe                   -0.0262
net_sharpe                      0.0005
max_drawdown                    0.0111
average_rebalance_turnover     -0.0971
total_transaction_cost         -0.0561
average_daily_gross_exposure    0.0002
Name: composite_minus_equal_exposure_sleeves, dtype: float64

In [21]:
controlled_gross_returns = pd.concat(
    {
        portfolio_name: daily.set_index("date")["gross_return"]
        for portfolio_name, daily
        in controlled_comparison_daily.items()
    },
    axis=1,
)

controlled_gross_returns.corr().round(4)

,Original Independent Sleeves,Equal-Exposure Independent Sleeves,Composite Score
Original Independent Sleeves,1.0000,0.9858,0.9457
Equal-Exposure Independent Sleeves,0.9858,1.0000,0.9501
Composite Score,0.9457,0.9501,1.0000


#### Measure the remaining construction difference

In [22]:
target_pair = (
    controlled_sleeve_targets[
        ["date", "ticker", "weight"]
    ]
    .rename(columns={"weight": "sleeve_weight"})
    .merge(
        composite_targets[
            ["date", "ticker", "weight"]
        ].rename(
            columns={"weight": "composite_weight"}
        ),
        on=["date", "ticker"],
        how="outer",
    )
    .fillna(
        {
            "sleeve_weight": 0.0,
            "composite_weight": 0.0,
        }
    )
)

target_structure_rows = []

for date, date_targets in target_pair.groupby("date"):
    sleeve_weight = date_targets["sleeve_weight"]
    composite_weight = date_targets["composite_weight"]

    target_gross = composite_weight.abs().sum()

    long_overlap = np.minimum(
        sleeve_weight.clip(lower=0.0),
        composite_weight.clip(lower=0.0),
    ).sum()

    short_overlap = np.minimum(
        (-sleeve_weight.clip(upper=0.0)),
        (-composite_weight.clip(upper=0.0)),
    ).sum()

    same_side_overlap = long_overlap + short_overlap

    target_structure_rows.append(
        {
            "date": date,
            "target_gross": target_gross,
            "l1_weight_difference": (
                sleeve_weight - composite_weight
            ).abs().sum(),
            "same_side_weight_overlap": same_side_overlap,
            "same_side_overlap_fraction": (
                same_side_overlap / target_gross
                if target_gross > 0
                else np.nan
            ),
        }
    )

target_structure = pd.DataFrame(target_structure_rows)

target_structure[
    [
        "l1_weight_difference",
        "same_side_weight_overlap",
        "same_side_overlap_fraction",
    ]
].agg(["mean", "std", "min", "max"]).round(4)

,l1_weight_difference,same_side_weight_overlap,same_side_overlap_fraction
mean,1.2697,1.1887,0.6518
std,0.4579,0.3877,0.0605
min,0.0000,0.0000,0.3750
max,2.5000,1.6500,0.8250


### Experiment 3: Controlled gross-exposure comparison

To separate portfolio-construction effects from exposure effects, the independent sleeve targets were rescaled on each rebalance date to match the composite portfolio's target gross exposure. The exposure audit confirms an exact match, with average target gross exposure of approximately 1.82 for both portfolios.

At matched exposure, the independent sleeves produce a slightly higher gross annualised return (15.53% versus 15.19%) and gross Sharpe ratio (0.823 versus 0.797). The composite portfolio, however, has lower average rebalance turnover (0.477 versus 0.574) and lower transaction costs.

Consequently, their net performance is effectively identical: net Sharpe ratios are 0.678 for the independent sleeves and 0.679 for the composite. The composite also has a modestly smaller maximum drawdown (-30.63% versus -31.73%).

The two portfolios have a return correlation of approximately 0.95, while their average same-side target-weight overlap is approximately 65%. They therefore express broadly similar factor information but retain meaningful differences in stock selection and weighting.

Overall, neither construction method is decisively superior at matched exposure. The composite score is marginally more implementation-efficient, while the independent sleeves preserve clearer factor-level attribution. The original sleeve portfolio's smoother risk profile primarily results from natural netting and lower realised gross exposure.